In [ ]:
# Import the libraries required for environment variables, JSON handling, and the OpenAI client.
import os
import json

from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# Load the OpenAI API key and initialize the OpenAI client.
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found, kindly recheck this.")
else:
    print("API key found.")

openai = OpenAI()

In [ ]:
# Define a realistic business invoice that the document intelligence system will analyze.
document = """
INVOICE

Invoice Number: INV-2026-0042

Vendor:
FlowTech Automation Ltd.
12 Marina Road, Lagos, Nigeria

Customer:
Green Logistics Nigeria Ltd.
25 Airport Road, Port Harcourt, Nigeria

Invoice Date: September 10, 2026
Due Date: October 10, 2026

Payment Terms: Net 30

Currency: USD

Items:

1. AI Workflow Automation Setup
Quantity: 1
Unit Price: $1,500
Total: $1,500

2. Monthly Automation Support
Quantity: 2
Unit Price: $500
Total: $1,000

Subtotal: $2,500
Tax: $187.50
Total: $2,687.50
"""

In [ ]:
# Define the instructions for extracting structured information from the business document.
document_prompt = """
You are a document intelligence assistant.

Analyze the document and return valid JSON using exactly this structure:

{
    "document_type": "",
    "invoice_number": "",
    "vendor": "",
    "customer": "",
    "invoice_date": "",
    "due_date": "",
    "currency": "",
    "payment_terms": "",
    "line_items": [],
    "subtotal": "",
    "tax": "",
    "total": ""
}

For line_items, use this structure:

[
    {
        "description": "",
        "quantity": "",
        "unit_price": "",
        "total": ""
    }
]

Only extract information explicitly present in the document.

Do not invent information.

If information is not available, use an empty string or empty list.

Return only valid JSON.
"""

In [ ]:
# Define the required document fields and validate the structured response returned by the LLM.
document_fields = [
    "document_type",
    "invoice_number",
    "vendor",
    "customer",
    "invoice_date",
    "due_date",
    "currency",
    "payment_terms",
    "line_items",
    "subtotal",
    "tax",
    "total"
]


def validate_document(data):
    if not data:
        return False

    return all(field in data for field in document_fields)

In [ ]:
# Send the document to the LLM, parse the JSON response, and validate the extracted information.
def analyze_document(document):
    messages = [
        {"role": "system", "content": document_prompt},
        {"role": "user", "content": document}
    ]

    try:
        response = openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages
        )

        result = response.choices[0].message.content
        data = json.loads(result)

        if not validate_document(data):
            print("The LLM response is missing required fields.")
            return None

        return data

    except json.JSONDecodeError:
        print("The LLM returned invalid JSON.")
        return None

    except Exception as error:
        print(f"An error occurred: {error}")
        return None

In [ ]:
# Run the document analyzer and display the extracted invoice information.
result = analyze_document(document)

for field, value in result.items():
    print(f"\n{field.upper()}:")
    print(value)

In [ ]:
# Analyze the extracted invoice information and calculate the relationship between its financial values.
def analyze_invoice(data):
    subtotal = float(data["subtotal"].replace(",", "").replace("$", ""))
    tax = float(data["tax"].replace(",", "").replace("$", ""))
    total = float(data["total"].replace(",", "").replace("$", ""))

    calculated_total = subtotal + tax

    return {
        "subtotal": subtotal,
        "tax": tax,
        "total": total,
        "calculated_total": calculated_total,
        "totals_match": calculated_total == total
    }


invoice_analysis = analyze_invoice(result)

for field, value in invoice_analysis.items():
    print(f"{field.upper()}: {value}")

In [ ]:
# Create evaluation documents with known invoice numbers and totals for testing the extraction pipeline.
evaluation_documents = [
    {
        "expected_invoice": "INV-001",
        "expected_total": "1180",
        "document": """
        INVOICE
        Invoice Number: INV-001
        Vendor: Alpha Systems
        Customer: Beta Logistics
        Invoice Date: September 1, 2026
        Due Date: October 1, 2026
        Currency: USD
        Subtotal: $1,000
        Tax: $180
        Total: $1,180
        """
    },
    {
        "expected_invoice": "INV-002",
        "expected_total": "2360",
        "document": """
        INVOICE
        Invoice Number: INV-002
        Vendor: Tech Solutions
        Customer: Green Retail
        Invoice Date: September 5, 2026
        Due Date: October 5, 2026
        Currency: USD
        Subtotal: $2,000
        Tax: $360
        Total: $2,360
        """
    },
    {
        "expected_invoice": "INV-003",
        "expected_total": "590",
        "document": """
        INVOICE
        Invoice Number: INV-003
        Vendor: Automation Hub
        Customer: Fast Delivery
        Invoice Date: September 8, 2026
        Due Date: October 8, 2026
        Currency: USD
        Subtotal: $500
        Tax: $90
        Total: $590
        """
    }
]

In [ ]:
# Inspect each evaluation result in detail so we can identify exactly why the document accuracy is 66.67%.
correct = 0
failed = 0

for item in evaluation_documents:
    result = analyze_document(item["document"])

    print(f"\nExpected Invoice: {item['expected_invoice']}")
    print(f"Expected Total: {item['expected_total']}")

    if result is None:
        print("Predicted: FAILED")
        failed += 1
        print("-" * 50)
        continue

    print(f"Predicted Invoice: {result['invoice_number']}")
    print(f"Predicted Total: {result['total']}")

    invoice_correct = result["invoice_number"] == item["expected_invoice"]

    predicted_total = (
        result["total"]
        .replace(",", "")
        .replace("$", "")
        .strip()
    )

    expected_total = item["expected_total"].strip()

    total_correct = predicted_total == expected_total

    print(f"Invoice Correct: {invoice_correct}")
    print(f"Total Correct: {total_correct}")

    if invoice_correct and total_correct:
        correct += 1

    print("-" * 50)

accuracy = correct / len(evaluation_documents)

print(f"Successful predictions: {correct}")
print(f"Failed predictions: {failed}")
print(f"Document Extraction Accuracy: {accuracy:.2%}")